In [1]:
from data_engineering.flights.flight_type import get_flights, get_flights_from_iceberg
from datetime import date,timedelta
import polars as pl

start_date = date(2025,1,1)
end_date = date(2025,12,1)
df_flights = pl.DataFrame()
for target_date in [start_date + timedelta(days=i) for i in range((end_date - start_date).days)]:
    df_flights = pl.concat([df_flights, get_flights(target_date)], how="vertical")
df_flights

icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""0100a2""","""MSR524""","""SU-GCE""",2025-01-01 02:10:26.490,"""HE16""",2025-01-01 06:28:35.710,"""EBOS""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
"""0100a2""","""MSR525""","""SU-GCE""",2025-01-01 08:30:50.210,"""EBOS""",2025-01-01 12:14:43.040,"""HE42""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
"""0100a2""","""MSR520""","""SU-GCE""",2025-01-01 21:42:28.070,"""HE16""",2025-01-02 01:56:20.490,"""EBOS""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
"""0100a4""","""MSR662""","""SU-GCG""",2025-01-01 10:30:50.250,"""HETR""",2025-01-01 10:48:51.110,"""HE45""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
"""0100a4""","""MSR672""","""SU-GCG""",2025-01-01 17:50:34.320,"""EG-0057""",2025-01-01 17:58:34.810,"""EG-AUE""",false,false,false,false,"""A332""",null,"""AIRBUS A-330-200""","""A5"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""e94c8f""","""BOV700""","""CP-3215""",2025-11-30 04:22:34.980,"""AR-0402""",2025-11-30 05:00:31.850,"""SAEZ""",false,false,false,false,"""B738""",null,"""BOEING 737-800""","""A3"""
"""e94c8f""","""BOV701""","""CP-3215""",2025-11-30 06:19:37.140,"""SAEZ""",2025-11-30 06:53:50.400,"""AR-0402""",false,false,false,false,"""B738""",null,"""BOEING 737-800""","""A3"""
"""e94c8f""","""BOV736""","""CP-3215""",2025-11-30 16:31:14.320,"""BR-0725""",2025-11-30 17:44:16.250,"""SBGR""",false,false,false,false,"""B738""",null,"""BOEING 737-800""","""A3"""


In [2]:
df_flights = df_flights.filter((pl.col("ladd") == True) | (pl.col("pia") == True))

In [3]:
df_flights = df_flights.with_columns(
    pl.concat_str(
        [
            pl.col("icao"),
            pl.col("takeoff_time").dt.strftime("%Y-%m-%d %H:%M:%S"),
        ],
        separator="_",
    ).alias("flight_id")
)

In [39]:
from dataclasses import dataclass
from data_engineering.flights.flight_type import FLIGHT_POLARS_SCHEMA, Flight

@dataclass
class FlightObject:
    current_flight: Flight
    next_flights: list[Flight]
flight_objects = []
for flight in df_flights.iter_rows(named=True):
    flight = Flight(**flight)
    filter_expr = ((pl.col("takeoff_time") > flight.landing_time) & ((pl.col("icao") == flight.icao) | (pl.col("pia") == True)) & (pl.col("takeoff_airport_ident") == flight.landing_airport_ident))
    df = df_flights.filter(filter_expr)
    next_flights = [Flight(**row) for row in df.iter_rows(named=True)]
    flight_object = FlightObject(current_flight=flight, next_flights=next_flights)
    flight_objects.append(flight_object)

KeyboardInterrupt: 

In [ ]:
flight_objects[0]

In [ ]:
# how do I deal with getting back to the icaos. 
# I can iterate for through each flight
# I'm going to try with assuuming flight data is perfect first just to see how far I get. 